# Model Comparison Analysis: Learnable vs Bayesian vs RGCN

This notebook analyzes and visualizes the performance metrics for three different models on the OntoOmicsKG dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

In [3]:
# Load the data
df = pd.read_csv('model_comparison_metrics.csv')
print(f"Loaded {len(df)} rows")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nModels: {df['Model'].unique()}")
print(f"\nFirst few rows:")
df.head(10)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 4, saw 2


In [4]:
# Separate overall and per-relation metrics
overall_df = df[df['Relation'] == 'Overall'].copy()
relations_df = df[df['Relation'] != 'Overall'].copy()

print("Overall Metrics:")
print(overall_df[['Model', 'MRR', 'MR', 'Hits@1', 'Hits@3', 'Hits@10']])
print(f"\nPer-relation metrics: {len(relations_df)} rows")

NameError: name 'df' is not defined

## 1. Overall Performance Comparison

In [ ]:
# Overall metrics comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Overall Performance Comparison: Learnable vs Bayesian vs RGCN', fontsize=16, fontweight='bold')

metrics = ['MRR', 'MR', 'Hits@1', 'Hits@3', 'Hits@10']
colors = ['#2E86AB', '#A23B72', '#F18F01']
models = ['learnable', 'bayesian', 'rgcn']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    
    values = [overall_df[overall_df['Model'] == model][metric].values[0] for model in models]
    bars = ax.bar(models, values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}',
                ha='center', va='bottom', fontweight='bold')
    
    ax.set_title(f'{metric}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=10)
    ax.set_ylim(0, max(values) * 1.2)
    ax.grid(axis='y', alpha=0.3)

# Remove empty subplot
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.show()

## 2. Per-Relation Performance Analysis

In [ ]:
# Create a mapping for consistent relation ordering
# Use the learnable/bayesian order as reference
relation_order = [
    'isAssociatedWithGO',
    'hasGeneExpressionOA',
    'participatedIn',
    'partOfPathway',
    'hasPhysicalInteractionWith',
    'isAssociatedWithProteinPathway',
    'containedIn',
    'hasModification',
    'hasFunctionalInteractionWith',
    'isAGO',
    'hasGeneticInteractionWith',
    'regulatesGO',
    'negativelyRegulatesGO',
    'occursInGO',
    'partOfGO',
    'hasPartGO',
    'positivelyRegulatesGO'
]

# Create a pivot table for easier comparison
mrr_pivot = relations_df.pivot_table(
    index='RelationName', 
    columns='Model', 
    values='MRR',
    aggfunc='first'
).reindex(relation_order)

print("MRR Comparison by Relation:")
print(mrr_pivot)

In [ ]:
# MRR comparison across relations
fig, ax = plt.subplots(figsize=(16, 8))

x = np.arange(len(relation_order))
width = 0.25

learnable_mrr = [mrr_pivot.loc[rel, 'learnable'] if 'learnable' in mrr_pivot.columns else 0 for rel in relation_order]
bayesian_mrr = [mrr_pivot.loc[rel, 'bayesian'] if 'bayesian' in mrr_pivot.columns else 0 for rel in relation_order]
rgcn_mrr = [mrr_pivot.loc[rel, 'rgcn'] if 'rgcn' in mrr_pivot.columns else 0 for rel in relation_order]

bars1 = ax.bar(x - width, learnable_mrr, width, label='Learnable', color='#2E86AB', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x, bayesian_mrr, width, label='Bayesian', color='#A23B72', alpha=0.8, edgecolor='black')
bars3 = ax.bar(x + width, rgcn_mrr, width, label='RGCN', color='#F18F01', alpha=0.8, edgecolor='black')

ax.set_xlabel('Relation Type', fontsize=12, fontweight='bold')
ax.set_ylabel('MRR', fontsize=12, fontweight='bold')
ax.set_title('MRR Comparison Across Relations', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(relation_order, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Hits@10 comparison
hits10_pivot = relations_df.pivot_table(
    index='RelationName', 
    columns='Model', 
    values='Hits@10',
    aggfunc='first'
).reindex(relation_order)

fig, ax = plt.subplots(figsize=(16, 8))

learnable_h10 = [hits10_pivot.loc[rel, 'learnable'] if 'learnable' in hits10_pivot.columns else 0 for rel in relation_order]
bayesian_h10 = [hits10_pivot.loc[rel, 'bayesian'] if 'bayesian' in hits10_pivot.columns else 0 for rel in relation_order]
rgcn_h10 = [hits10_pivot.loc[rel, 'rgcn'] if 'rgcn' in hits10_pivot.columns else 0 for rel in relation_order]

bars1 = ax.bar(x - width, learnable_h10, width, label='Learnable', color='#2E86AB', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x, bayesian_h10, width, label='Bayesian', color='#A23B72', alpha=0.8, edgecolor='black')
bars3 = ax.bar(x + width, rgcn_h10, width, label='RGCN', color='#F18F01', alpha=0.8, edgecolor='black')

ax.set_xlabel('Relation Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Hits@10', fontsize=12, fontweight='bold')
ax.set_title('Hits@10 Comparison Across Relations', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(relation_order, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Performance vs Relation Frequency

In [ ]:
# Get relation counts (use learnable as reference)
relation_counts = relations_df[relations_df['Model'] == 'learnable'][['RelationName', 'Count']].drop_duplicates()
relation_counts = relation_counts.set_index('RelationName').reindex(relation_order)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('MRR vs Relation Frequency', fontsize=14, fontweight='bold')

for idx, model in enumerate(['learnable', 'bayesian', 'rgcn']):
    ax = axes[idx]
    
    model_data = relations_df[relations_df['Model'] == model]
    model_data = model_data.set_index('RelationName').reindex(relation_order)
    
    counts = relation_counts['Count'].values
    mrrs = model_data['MRR'].values
    
    scatter = ax.scatter(counts, mrrs, s=100, alpha=0.7, edgecolors='black', linewidth=1.5)
    
    # Add relation labels for small counts
    for i, rel in enumerate(relation_order):
        if counts[i] < 100:  # Label relations with < 100 examples
            ax.annotate(rel[:15], (counts[i], mrrs[i]), 
                       fontsize=7, alpha=0.7, rotation=45)
    
    ax.set_xlabel('Relation Count (log scale)', fontsize=10, fontweight='bold')
    ax.set_ylabel('MRR', fontsize=10, fontweight='bold')
    ax.set_title(f'{model.capitalize()}', fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Heatmap Comparison

In [ ]:
# Create heatmap for MRR
fig, ax = plt.subplots(figsize=(10, 10))

heatmap_data = mrr_pivot[['learnable', 'bayesian', 'rgcn']]
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', 
            cbar_kws={'label': 'MRR'}, linewidths=0.5, linecolor='gray',
            ax=ax, vmin=0, vmax=0.7)

ax.set_title('MRR Heatmap: Model Performance by Relation', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Relation Type', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Improvement Analysis

In [ ]:
# Calculate improvement of Bayesian over Learnable
learnable_mrr_dict = dict(zip(
    relations_df[relations_df['Model'] == 'learnable']['RelationName'],
    relations_df[relations_df['Model'] == 'learnable']['MRR']
))

bayesian_mrr_dict = dict(zip(
    relations_df[relations_df['Model'] == 'bayesian']['RelationName'],
    relations_df[relations_df['Model'] == 'bayesian']['MRR']
))

improvements = []
for rel in relation_order:
    if rel in learnable_mrr_dict and rel in bayesian_mrr_dict:
        imp = bayesian_mrr_dict[rel] - learnable_mrr_dict[rel]
        improvements.append({
            'Relation': rel,
            'Improvement': imp,
            'Learnable_MRR': learnable_mrr_dict[rel],
            'Bayesian_MRR': bayesian_mrr_dict[rel]
        })

improvements_df = pd.DataFrame(improvements)
improvements_df = improvements_df.sort_values('Improvement', ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))

colors_bar = ['green' if x > 0 else 'red' for x in improvements_df['Improvement']]
bars = ax.barh(improvements_df['Relation'], improvements_df['Improvement'], 
               color=colors_bar, alpha=0.7, edgecolor='black')

ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('MRR Improvement (Bayesian - Learnable)', fontsize=12, fontweight='bold')
ax.set_title('Bayesian vs Learnable: Per-Relation MRR Improvement', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTop 5 Relations with Best Improvement:")
print(improvements_df.head(5)[['Relation', 'Improvement', 'Learnable_MRR', 'Bayesian_MRR']])

## 6. Summary Statistics

In [ ]:
# Summary statistics
summary_stats = relations_df.groupby('Model').agg({
    'MRR': ['mean', 'std', 'min', 'max'],
    'MR': ['mean', 'std', 'min', 'max'],
    'Hits@1': ['mean', 'std', 'min', 'max'],
    'Hits@3': ['mean', 'std', 'min', 'max'],
    'Hits@10': ['mean', 'std', 'min', 'max']
}).round(4)

print("Summary Statistics by Model:")
print(summary_stats)

# Create a visual summary
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Summary Statistics: Mean Performance by Model', fontsize=16, fontweight='bold')

metrics = ['MRR', 'MR', 'Hits@1', 'Hits@3', 'Hits@10']
for idx, metric in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    
    means = [summary_stats.loc[model, (metric, 'mean')] for model in models]
    stds = [summary_stats.loc[model, (metric, 'std')] for model in models]
    
    bars = ax.bar(models, means, yerr=stds, color=colors, alpha=0.8, 
                  edgecolor='black', linewidth=1.5, capsize=5)
    
    for bar, mean_val in zip(bars, means):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{mean_val:.4f}',
                ha='center', va='bottom', fontweight='bold')
    
    ax.set_title(f'{metric} (Mean ± Std)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

fig.delaxes(axes[1, 2])
plt.tight_layout()
plt.show()

## 7. Relation Categories Analysis

In [ ]:
# Categorize relations by type
def categorize_relation(rel_name):
    rel_lower = rel_name.lower()
    if 'go' in rel_lower:
        return 'GO-related'
    elif 'pathway' in rel_lower:
        return 'Pathway-related'
    elif 'interaction' in rel_lower:
        return 'Interaction'
    elif 'modification' in rel_lower:
        return 'Modification'
    elif 'participated' in rel_lower:
        return 'Participation'
    elif 'contained' in rel_lower or 'part' in rel_lower:
        return 'Containment/Part'
    elif 'expression' in rel_lower:
        return 'Expression'
    else:
        return 'Other'

relations_df['Category'] = relations_df['RelationName'].apply(categorize_relation)

# Average MRR by category
category_stats = relations_df.groupby(['Model', 'Category'])['MRR'].mean().reset_index()
category_pivot = category_stats.pivot(index='Category', columns='Model', values='MRR')

fig, ax = plt.subplots(figsize=(12, 8))
category_pivot.plot(kind='bar', ax=ax, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_title('Average MRR by Relation Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Category', fontsize=12, fontweight='bold')
ax.set_ylabel('Average MRR', fontsize=12, fontweight='bold')
ax.legend(title='Model', fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAverage MRR by Category:")
print(category_pivot)